# Polynormer: a polynomial-expressive graph transformer

This tutorial introduces **Polynormer** ([Deng, Yue & Zhang, ICLR 2024](https://arxiv.org/abs/2403.01232)), a graph transformer that learns a high-degree *equivariant polynomial* on the node features whose coefficients are produced by attention. It is integrated into TopoBench as the graph backbone `topobench.nn.backbones.Polynormer`.

Polynormer is built from two modules applied in sequence:

1. **Local equivariant attention** (Eq. 7). Each layer mixes a node with its graph neighbours via a GAT-style sparse attention and applies the gated polynomial update
$$X \leftarrow (1-\beta)\,\mathrm{LN}(H \odot X) + \beta\, X, \qquad X = \mathrm{GAT}(X, E) + W X, \quad H = \mathrm{ReLU}(W_h X).$$
The Hadamard product $H \odot X$ injects a degree-2 interaction per layer, so $L$ stacked layers reach degree-$2^L$ polynomial expressivity (Thm. 3.3).

2. **Global linear attention** (Eq. 6 & 8). A kernel-trick attention with a sigmoid feature map $\sigma$,
$$\mathrm{num}=\sigma(Q)\,(\sigma(K)^\top V), \qquad \mathrm{den}=\sigma(Q)\,\textstyle\sum_i \sigma(K_{i,:}),$$
mixes all nodes in **linear** $O(Nd^2)$ time (the $N\times N$ attention matrix is never formed).

**TopoBench adaptation.** The reference implementation runs on a single graph. TopoBench feeds *mini-batches of disjoint graphs*, so our global attention is made **batch-aware**: the kernel sums are accumulated per graph (segment-wise over the `batch` vector), so nodes never attend across graph boundaries. For a single graph this is identical to the paper. We verify this property numerically below.

## 1. The backbone

The backbone maps node features `[num_nodes, in_channels]` to embeddings `[num_nodes, out_channels]`; the final task logits are produced by the TopoBench readout. We build a small instance and run it on a synthetic graph.

In [ ]:
import torch
from torch_geometric.data import Batch, Data

from topobench.nn.backbones.graph.polynormer import Polynormer

torch.manual_seed(0)

model = Polynormer(
    in_channels=16,
    hidden_channels=32,
    out_channels=16,
    local_layers=3,
    global_layers=2,
    heads=1,
    beta=-1.0,  # learnable per-layer gates
).eval()

n = 12
x = torch.randn(n, 16)
edge_index = torch.randint(0, n, (2, 36))
out = model(x, edge_index, batch=torch.zeros(n, dtype=torch.long))
print('output shape:', tuple(out.shape))
print('parameters  :', sum(p.numel() for p in model.parameters()))

## 2. Local-only vs. local-to-global

Setting `global_layers=0` gives the faithful **local-only** Polynormer variant. Comparing it to the full model shows that the global module is wired in only when requested (it adds long-range, all-pairs mixing on top of the local message passing).

In [ ]:
local_only = Polynormer(16, 32, 16, local_layers=3, global_layers=0).eval()
local_plus_global = Polynormer(16, 32, 16, local_layers=3, global_layers=2).eval()

print('local-only uses global module :', local_only.use_global)
print('full model uses global module :', local_plus_global.use_global)

## 3. The global attention is batch-aware

This is the key correctness property for TopoBench. We run the model on a graph **A** on its own, then on a batch **[A, B]**, and check that A's node embeddings are unchanged. If the global attention leaked across graphs, batching B in would perturb A's outputs.

In [ ]:
g_a = Data(x=torch.randn(9, 16), edge_index=torch.randint(0, 9, (2, 24)), num_nodes=9)
g_b = Data(x=torch.randn(15, 16), edge_index=torch.randint(0, 15, (2, 40)), num_nodes=15)

out_a_alone = model(g_a.x, g_a.edge_index, batch=torch.zeros(9, dtype=torch.long))

batch = Batch.from_data_list([g_a, g_b])
out_batched = model(batch.x, batch.edge_index, batch=batch.batch)
out_a_in_batch = out_batched[:9]

max_diff = (out_a_alone - out_a_in_batch).abs().max().item()
print(f'max |A_alone - A_in_batch| = {max_diff:.2e}  (almost 0 => no cross-graph leakage)')

## 4. Training Polynormer in the TopoBench pipeline

The model ships with the Hydra config `configs/model/graph/polynormer.yaml`, so the canonical way to train it is `python -m topobench model=graph/polynormer dataset=<dataset>`. Here we run a short training on the MUTAG graph-classification dataset directly from the notebook.

In [ ]:
import os
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from topobench.run import run
from topobench.utils.config_resolvers import register_all_resolvers

# Locate the repository root (the directory containing configs/run.yaml).
root = Path.cwd().resolve()
while not (root / 'configs' / 'run.yaml').exists():
    root = root.parent
os.environ['PROJECT_ROOT'] = str(root)

out_dir = root / 'logs' / 'tutorial_polynormer'
out_dir.mkdir(parents=True, exist_ok=True)

register_all_resolvers()
GlobalHydra.instance().clear()
with initialize_config_dir(version_base='1.3', config_dir=str(root / 'configs')):
    cfg = compose(
        config_name='run.yaml',
        overrides=[
            'model=graph/polynormer',
            'dataset=graph/MUTAG',
            'logger=csv',
            'trainer.max_epochs=10',
            'trainer.accelerator=cpu',
            'trainer.devices=1',
            '+trainer.enable_progress_bar=False',
            f'paths.output_dir={out_dir.as_posix()}',
            f'paths.work_dir={root.as_posix()}',
        ],
    )

metric_dict, _ = run(cfg)
print('\nMetrics:')
for key, value in metric_dict.items():
    print(f'{key:<20s} {float(value):.4f}')

That's it. We built the Polynormer backbone, saw its local-to-global structure, verified the batch-aware global attention, and trained it end-to-end through the standard TopoBench pipeline. To benchmark it on the challenge's GraphUniverse datasets, set `MODEL_CONFIG = "graph/polynormer"` in `2026_tdl_challenge/run_evaluation.ipynb`.